The Nobel Prize has been among the most prestigious international awards since 1901. Each year, awards are bestowed in chemistry, literature, physics, physiology or medicine, economics, and peace. In addition to the honor, prestige, and substantial prize money, the recipient also gets a gold medal with an image of Alfred Nobel (1833 - 1896), who established the prize.

![](Nobel_Prize.png)

The Nobel Foundation has made a dataset available of all prize winners from the outset of the awards from 1901 to 2023. The dataset used in this project is from the Nobel Prize API and is available in the `nobel.csv` file in the `data` folder.

In this project, you'll get a chance to explore and answer several questions related to this prizewinning data. And we encourage you then to explore further questions that you're interested in!

# Question 1: What is the most commonly awarded gender and birth country?

In [291]:
# Loading in required libraries
import pandas as pd
import seaborn as sns
import numpy as np
import geopandas as gpd

# Preliminary: Import the data
nobel = pd.read_csv("./data/nobel.csv")
nobel.head()

,year,category,prize,motivation,prize_share,laureate_id,laureate_type,full_name,birth_date,birth_city,birth_country,sex,organization_name,organization_city,organization_country,death_date,death_city,death_country
0,1901,Chemistry,The Nobel Prize in Chemistry 1901,"""in recognition of the extraordinary services ...",1/1,160,Individual,Jacobus Henricus van 't Hoff,1852-08-30,Rotterdam,Netherlands,Male,Berlin University,Berlin,Germany,1911-03-01,Berlin,Germany
1,1901,Literature,The Nobel Prize in Literature 1901,"""in special recognition of his poetic composit...",1/1,569,Individual,Sully Prudhomme,1839-03-16,Paris,France,Male,NaN,NaN,NaN,1907-09-07,Châtenay,France
2,1901,Medicine,The Nobel Prize in Physiology or Medicine 1901,"""for his work on serum therapy, especially its...",1/1,293,Individual,Emil Adolf von Behring,1854-03-15,Hansdorf (Lawice),Prussia (Poland),Male,Marburg University,Marburg,Germany,1917-03-31,Marburg,Germany
3,1901,Peace,The Nobel Peace Prize 1901,NaN,1/2,462,Individual,Jean Henry Dunant,1828-05-08,Geneva,Switzerland,Male,NaN,NaN,NaN,1910-10-30,Heiden,Switzerland
4,1901,Peace,The Nobel Peace Prize 1901,NaN,1/2,463,Individual,Frédéric Passy,1822-05-20,Paris,France,Male,NaN,NaN,NaN,1912-06-12,Paris,France


## Remarks
- It's worth noting that the question asks about gender, but the dataset includes _sex_. This is unlikely to cause any significant issues though
- Dataset includes birth country separately from "organization_country". We want the _birth country_ of laureates.

In [292]:
# Don't overthink this! It's statistics! We just want the mode!

top_gender = nobel['sex'].mode()[0]
top_country = nobel['birth_country'].mode()[0]
print(f"Most common gender/sex is {top_gender}, and the most common country is {top_country}")
# Idea for extension: Show a map of the world with a gradient indicating frequencies

Most common gender/sex is Male, and the most common country is United States of America


# Question 2: Which decade had the highest ratio of US-born Nobel Prize winners to total winners?

In [293]:
# Find decade for each year example -> Ambiguity here, would "1956" be in the "50s" or "1950s"? Assume latter
def find_decade(year):
    return year//10*10
    
nobel['decade'] = nobel['year'].agg(find_decade)

In [294]:
# Subset by decade, find normalized count of each country and then select just the USA
prop_american = []
for decade in nobel.decade.unique():
    prop_american.append(nobel[nobel.loc[:,'decade']==decade]["birth_country"].value_counts(normalize=True).loc["United States of America"])

prop_american_df = pd.DataFrame(data=prop_american, index=nobel.decade.unique(),columns=["Proportion"])
max_decade_usa = int(prop_american_df.sort_values(by=['Proportion'],ascending=False).reset_index().iloc[0]["index"])
print(f"Most American decade was {max_decade_usa} with {prop_american_df.iloc[prop_american_df.index.get_loc(max_decade_usa)]['Proportion']*100:.2f}% of winners being American")

Most American decade was 2000 with 43.70% of winners being American


# Question 3: Which decade and category had the highest proportion of female Laureates?

In [295]:
# Calculate the proportion of female laureates per category per decade
female_proportion = (
    nobel.groupby(['decade', 'category'])['sex']
    .apply(lambda x: (x == 'Female').mean())
    .reset_index(name='female_proportion')
)

In [296]:
# Find the row with the highest female_proportion
top_row = female_proportion.sort_values(by="female_proportion", ascending=False).iloc[0]
max_female_dict = {
    int(top_row['decade']): top_row['category']
}
max_female_dict

{2020: 'Literature'}

# Question 4: First woman to win, and what category?

In [297]:
# Sort dataset by year
sorted_df = nobel.sort_values(by="year").reset_index(drop=True)
first_index = (sorted_df["sex"]=="Female").idxmax()
first_woman_name, first_woman_category = sorted_df.iloc[first_index][["full_name","category"]]
print(f"First Woman's Name was {first_woman_name}, in the {first_woman_category} category")

First Woman's Name was Marie Curie, née Sklodowska, in the Physics category


# Question 5: Who has won multiple times?

In [298]:
# Fix: Find duplicated full_names and filter the DataFrame accordingly
filter = nobel["full_name"].duplicated(keep=False)
repeat_list = nobel[filter]["full_name"].unique().tolist()
print(repeat_list)

['Marie Curie, née Sklodowska', 'Comité international de la Croix Rouge (International Committee of the Red Cross)', 'Linus Carl Pauling', 'Office of the United Nations High Commissioner for Refugees (UNHCR)', 'John Bardeen', 'Frederick Sanger']
